# Análisis de estaciones (TP3)

Mismo análisis que `TP2/notebooks/analisis_estaciones.ipynb`, aplicado al nuevo dataset de TP3
(`pronosticos-observaciones.csv` + `metadata_estaciones.json`). El objetivo es comparar los resultados
entre ambos: ¿la nueva muestra de estaciones corrige los problemas de selección que se encontraron en TP2?

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd


In [2]:
RAW = '../data/raw/'
PROCESSED = '../data/processed/'

estaciones_raw = RAW + 'metadata_estaciones.json'
estados_raw = RAW + 'US_StateBoundaries.geojson'

# Misma lista de 15 estados que en TP2
estados_central_abrev = [
    "ND", "SD", "NE", "KS", "OK", "TX",
    "MN", "IA", "MO", "AR", "LA",
    "WI", "IL", "MI", "IN"
]

estados_central_nombres = [
    "North Dakota", "South Dakota", "Nebraska", "Kansas", "Oklahoma", "Texas",
    "Minnesota", "Iowa", "Missouri", "Arkansas", "Louisiana",
    "Wisconsin", "Illinois", "Michigan", "Indiana"
]


## 1. Carga de datos

In [3]:
estaciones = pd.read_json(estaciones_raw)
estados = gpd.read_file(estados_raw)

print(f'Estaciones: {estaciones.shape[0]} filas, {estaciones.shape[1]} columnas')
print(f'Estados (polígonos): {estados.shape[0]} filas - CRS {estados.crs}')
estaciones.head()


Estaciones: 500 filas, 14 columnas
Estados (polígonos): 53 filas - CRS EPSG:4326


,id,elevation,nombre,county,forecast,fireWeatherZone,estado,timeZone,provider,subProvider,longitud,latitud,cobertura_horaria,dist_vecina_km
0,011CE,4.5720,51 Bacliff,NaN,https://api.weather.gov/zones/forecast/GMZ335,NaN,TX,America/Chicago,MesoWest,CENTERPOINT,-94.98226,29.51261,1.0,59.7
1,057CE,31.3944,01 Richmond James Pink,https://api.weather.gov/zones/county/TXC157,https://api.weather.gov/zones/forecast/TXZ227,https://api.weather.gov/zones/fire/TXZ227,TX,America/Chicago,MesoWest,CENTERPOINT,-95.77481,29.58948,1.0,52.9
2,AFWW3,295.6560,AUGUSTA,https://api.weather.gov/zones/county/WIC035,https://api.weather.gov/zones/forecast/WIZ028,https://api.weather.gov/zones/fire/WIZ028,WI,America/Chicago,RAWS,,-91.13446,44.69573,1.0,61.8
3,AGTN1,1341.1200,AGATE,https://api.weather.gov/zones/county/NEC165,https://api.weather.gov/zones/forecast/NEZ096,https://api.weather.gov/zones/fire/NEZ435,NE,America/Denver,RAWS,,-103.73583,42.42500,1.0,63.3
4,ARD02,0.0000,Hwy 82 - Ouachita River bridge Felsenthal -WLM,https://api.weather.gov/zones/county/ARC003,https://api.weather.gov/zones/forecast/ARZ074,https://api.weather.gov/zones/fire/ARZ074,AR,America/Chicago,MesoWest,ARDOT,-92.10386,33.15075,1.0,66.5


## 2. Filtrado de los estados de la región central

In [4]:
estados_central = (
    estados[estados['STATE_ABBR'].isin(estados_central_abrev)]
    [['NAME', 'STATE_ABBR', 'geometry']]
    .rename(columns={'NAME': 'nombre_estado', 'STATE_ABBR': 'estado'})
    .reset_index(drop=True)
)

# Control: que estén los 15 estados de la lista
faltantes = set(estados_central_abrev) - set(estados_central['estado'])
print(f'Estados seleccionados: {len(estados_central)} - faltantes: {faltantes or "ninguno"}')


Estados seleccionados: 15 - faltantes: ninguno


### Sin estaciones sin estado asignado

A diferencia de TP2 (donde 31 estaciones venían con `estado = "N/A"` y hubo que asignarles un estado por
polígono o por cercanía), en este dataset la metadata ya trae el estado resuelto para las 500 estaciones.
Se verifica abajo que no aparezca `"N/A"` y que todas caigan dentro de los 15 estados de la región central.

In [5]:
print(estaciones['estado'].value_counts().to_string())


estado
TX    114
MN     37
KS     36
NE     33
SD     33
ND     31
OK     30
MO     30
MI     25
WI     24
IL     24
IA     24
AR     23
LA     20
IN     16


In [6]:
CRS_METRICO = 'EPSG:5070'  # mismo que en TP2: Albers equivalente para EE.UU. contiguo

estaciones_gdf = gpd.GeoDataFrame(
    estaciones,
    geometry=gpd.points_from_xy(estaciones['longitud'], estaciones['latitud']),
    crs='EPSG:4326'
)
# Se agrega esta columna sólo para que el esquema coincida con el de TP2 (ahí distingue
# el estado asignado por polígono/cercanía del que ya venía en la metadata)
estaciones_gdf['origen_estado'] = 'dato original'

estaciones_central = (
    estaciones_gdf[estaciones_gdf['estado'].isin(estados_central_abrev)]
    .reset_index(drop=True)
)
print(f'Estaciones en la región central: {len(estaciones_central)} de {len(estaciones_gdf)}')


Estaciones en la región central: 500 de 500


## 3. Estaciones que no reportan observaciones

En TP2 esta sección leía `estaciones_sin_meteo.csv`, generado aparte por `ETL/clasificar_estaciones.py`.
Como se decidió no tocar `ETL/` para TP3, acá se repite la misma lógica de ese script directamente en el
notebook: por estación, qué fracción de las horas tiene dato en cada variable observada
(`obs_temp_c`, `obs_dewpoint_c`, `obs_rh_pct`, `obs_windspeed_kmh`, `obs_wind_direction_d`).

Categorías (mismos umbrales que el script: 1 % y 90 % de cobertura):

- **`sin_registros`**: la estación no tiene ninguna fila en el dataset.
- **`sin_meteo`**: tiene filas, pero nunca reporta ninguna variable meteorológica (cobertura máxima < 1 %).
- **`intermitente`**: reporta, pero con huecos frecuentes (cobertura máxima < 90 %).
- **`parcial_por_variable`**: cobertura alta, pero falta alguna variable puntual.
- **`ok`**: cobertura completa en todas las variables.

Se excluyen `sin_meteo` y `sin_registros` (igual que en TP2); las demás cuentan como "con datos".


In [7]:
VARIABLES_OBS = ['obs_temp_c', 'obs_dewpoint_c', 'obs_rh_pct', 'obs_windspeed_kmh', 'obs_wind_direction_d']
UMBRAL = 0.01              # cobertura mínima para considerar que una variable se mide
UMBRAL_INTERMITENTE = 0.90  # por debajo: intermitente

mediciones = pd.read_csv(RAW + 'pronosticos-observaciones.csv', usecols=['station_id', *VARIABLES_OBS], dtype=str)
print(f'{len(mediciones):,} filas, {mediciones["station_id"].nunique()} estaciones con al menos una fila')


130,562 filas, 498 estaciones con al menos una fila


In [8]:
agrupado = mediciones.groupby('station_id')
cobertura = agrupado[VARIABLES_OBS].agg(lambda s: s.notna().mean())
cobertura.columns = [f'cob_{c.removeprefix("obs_")}' for c in VARIABLES_OBS]
cobertura.insert(0, 'n_horas', agrupado.size())
cobertura['cobertura_max'] = cobertura.filter(like='cob_').max(axis=1)

# Left join desde la metadata (no desde las mediciones): así también aparecen
# las estaciones que no tienen ninguna fila en el dataset
cobertura = (
    estaciones_central[['id']].rename(columns={'id': 'station_id'})
    .merge(cobertura, on='station_id', how='left')
)
cobertura.head()


,station_id,n_horas,cob_temp_c,cob_dewpoint_c,cob_rh_pct,cob_windspeed_kmh,cob_wind_direction_d,cobertura_max
0,011CE,210.0,0.919048,0.919048,0.919048,0.890476,0.890476,0.919048
1,057CE,201.0,0.915423,0.915423,0.915423,0.915423,0.915423,0.915423
2,AFWW3,196.0,0.974490,0.974490,0.974490,0.974490,0.974490,0.974490
3,AGTN1,185.0,0.881081,0.935135,0.881081,0.945946,0.945946,0.945946
4,ARD02,201.0,0.895522,0.890547,0.890547,0.895522,0.895522,0.895522


In [9]:
def clasificar(fila, umbral=UMBRAL):
    if pd.isna(fila['n_horas']):
        return 'sin_registros', 'sin filas en el dataset'

    columnas_cob = [c for c in fila.index if c.startswith('cob_')]
    sin_dato = [c.removeprefix('cob_') for c in columnas_cob if fila[c] < umbral]

    if fila['cobertura_max'] < umbral:
        return 'sin_meteo', 'nunca reporta variables meteorológicas'
    if fila['cobertura_max'] < UMBRAL_INTERMITENTE:
        return 'intermitente', 'reporta, pero con huecos frecuentes'
    if sin_dato:
        return 'parcial_por_variable', 'no mide algunas variables'
    return 'ok', ''


cobertura[['categoria', 'motivo']] = cobertura.apply(lambda f: pd.Series(clasificar(f)), axis=1)
cobertura['categoria'].value_counts()


categoria
ok               443
intermitente      55
sin_registros      2
Name: count, dtype: int64

In [10]:
estaciones_central = (
    estaciones_central
    .merge(cobertura[['station_id', 'categoria']], left_on='id', right_on='station_id', how='left')
    .drop(columns='station_id')
)
# "con datos" = igual criterio que TP2: todo lo que no sea sin_meteo/sin_registros
# (acá se conserva la categoría fina -ok/intermitente/parcial_por_variable- en vez de
# colapsarla, porque no depende de un CSV externo)
estaciones_central['con_datos'] = ~estaciones_central['categoria'].isin(['sin_meteo', 'sin_registros'])

estaciones_validas = estaciones_central[estaciones_central['con_datos']].reset_index(drop=True)
print(f'Estaciones con observaciones: {len(estaciones_validas)} de {len(estaciones_central)}')

pd.crosstab(estaciones_central['provider'].replace('', '(sin proveedor)'), estaciones_central['categoria'],
            margins=True, margins_name='Total')


Estaciones con observaciones: 498 de 500


categoria,intermitente,ok,sin_registros,Total
provider,,,,
APRSWXNET,17,1,0,18
ASOS,1,13,0,14
ASOS-HFM,0,136,0,136
MARITIME,0,2,0,2
MesoWest,25,6,0,31
NonFedAWOS,5,48,1,54
OTHER-MTR,5,182,1,188
RAWS,2,55,0,57
Total,55,443,2,500


### Comparación con TP2

| | TP2 (selección alfabética) | TP3 (nueva muestra) |
|---|---|---|
| Estaciones con observaciones | 345 de 500 (69,0 %) | 498 de 500 (99,6 %) |
| `sin_meteo` (nunca miden) | 129 | 0 |
| `sin_registros` (sin filas) | 26 | 2 |
| Peor red afectada | HADS / hidrológicas (71) | MesoWest (25/31 intermitente), APRSWXNET (17/18 intermitente) |

La nueva muestra prácticamente elimina el problema que motivó esta sección en TP2: no incluye estaciones
puramente hidrológicas, así que no hay ninguna `sin_meteo`. Lo que persiste es la intermitencia de
**MesoWest** y, sobre todo, **APRSWXNET** (red de aficionados), que ya en `relaciones-bivariadas.ipynb`
de TP2 aparecía como la red que peor ajusta el viento — acá además reporta con más huecos.


## 4. Cantidad de estaciones por estado

Se cuentan sólo las estaciones con observaciones. La columna `excluidas` indica cuántas se descartaron en cada estado.

En TP2 la selección alfabética dejaba una densidad por estado muy desigual: de **0** estaciones cada 10.000 km²
en Dakota del Sur a **2,71** en Texas, que concentraba el 47,8 % de las estaciones originales (53,9 % tras excluir
las que no medían). Acá la densidad es prácticamente uniforme — entre **1,57 y 1,71** estaciones cada 10.000 km²
en los 15 estados, sin relación con su tamaño — y la exclusión de la sección 3 (sólo 2 estaciones) casi no mueve
los porcentajes: Texas pasa de 22,8 % a 22,7 %. Michigan, que en TP2 perdía más de la mitad de sus estaciones por
ser boyas de los Grandes Lagos, acá conserva las 25.


In [18]:
area_km2 = estados_central.set_index('estado').to_crs(CRS_METRICO).area / 1e6

excluidas_por_estado = estaciones_central[~estaciones_central['con_datos']].groupby('estado').size()

detalle_estados = (
    estaciones_validas.groupby('estado').size()
    .reindex(estados_central_abrev, fill_value=0)
    .rename('estaciones')
    .to_frame()
    .join(estados_central.set_index('estado')['nombre_estado'])
    .assign(
        excluidas=excluidas_por_estado.reindex(estados_central_abrev, fill_value=0),
        porcentaje=lambda d: (100 * d['estaciones'] / d['estaciones'].sum()).round(1),
        area_miles_km2=(area_km2 / 1000).round(1),
        estaciones_cada_10mil_km2=lambda d: (d['estaciones'] / area_km2 * 1e4).round(2),
    )
    .sort_values('estaciones', ascending=False)
    [['nombre_estado', 'estaciones', 'excluidas', 'porcentaje', 'area_miles_km2', 'estaciones_cada_10mil_km2']]
)
detalle_estados


,nombre_estado,estaciones,excluidas,porcentaje,area_miles_km2,estaciones_cada_10mil_km2
estado,,,,,,
TX,Texas,113,1,22.7,685.7,1.65
MN,Minnesota,37,0,7.4,218.5,1.69
KS,Kansas,36,0,7.2,213.1,1.69
SD,South Dakota,33,0,6.6,199.7,1.65
NE,Nebraska,33,0,6.6,200.3,1.65
ND,North Dakota,31,0,6.2,183.1,1.69
OK,Oklahoma,30,0,6.0,181.0,1.66
MO,Missouri,30,0,6.0,180.5,1.66
MI,Michigan,25,0,5.0,150.5,1.66


In [19]:
print(f"Estaciones con observaciones: {detalle_estados['estaciones'].sum()} "
      f"({detalle_estados['excluidas'].sum()} excluidas)")
print(f"Texas concentra el {detalle_estados.loc['TX', 'porcentaje']}% de las estaciones")
detalle_estados['estaciones'].describe().round(1)


Estaciones con observaciones: 498 (2 excluidas)
Texas concentra el 22.7% de las estaciones


count     15.0
mean      33.2
std       22.9
min       16.0
25%       24.0
50%       30.0
75%       33.0
max      113.0
Name: estaciones, dtype: float64

## 5. Ubicación geográfica de las estaciones

A diferencia de TP2, acá no hay boyas ni estaciones en el agua que requieran una marca aparte: todas las
estaciones tienen un estado asignado de origen (sección 2), así que el mapa sólo distingue entre estaciones
con datos y las excluidas en la sección 3.


In [20]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patheffects as pe

estados_plot = estados_central.to_crs(CRS_METRICO)
estaciones_plot = estaciones_validas.to_crs(CRS_METRICO)
excluidas_plot = estaciones_central[~estaciones_central['con_datos']].to_crs(CRS_METRICO)

AZUL, NARANJA, GRIS = '#2a78d6', '#eb6834', '#a3a29d'

fig, ax = plt.subplots(figsize=(10, 11))
estados_plot.plot(ax=ax, color='#ebeae6', edgecolor='white', linewidth=1.5)

excluidas_plot.plot(ax=ax, facecolor='none', edgecolor=GRIS, marker='o', markersize=14,
                    linewidth=0.8, zorder=2)

estaciones_plot.plot(ax=ax, color=AZUL, markersize=16, edgecolor='white', linewidth=0.5, zorder=3)

# Abreviatura y cantidad de estaciones en cada estado
for _, fila in estados_plot.iterrows():
    p = fila.geometry.representative_point()
    n = detalle_estados.loc[fila['estado'], 'estaciones']
    ax.annotate(f"{fila['estado']}\n{n}", (p.x, p.y), ha='center', va='center',
                fontsize=9, color='#52514e', fontweight='bold', zorder=4,
                path_effects=[pe.withStroke(linewidth=3, foreground='white')])

ax.legend(
    handles=[
        Line2D([], [], marker='o', ls='', color=AZUL, markersize=6, label='Estación con observaciones'),
        Line2D([], [], marker='o', ls='', markerfacecolor='none', markeredgecolor=GRIS, markersize=6,
               label=f'Excluida: sin observaciones útiles ({len(excluidas_plot)})'),
    ],
    loc='lower left', frameon=False, fontsize=9
)
ax.set_title(f'Ubicación de las {len(estaciones_plot)} estaciones con observaciones de la región central (TP3)\n'
             '(número bajo cada estado = cantidad de estaciones con datos)', fontsize=12, loc='center')
ax.set_axis_off()
plt.tight_layout()
plt.show()


C:\Users\usuario\AppData\Local\Temp\ipykernel_22640\1787162142.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Guardado de los datos filtrados

Se guardan las 500 estaciones con las columnas `categoria` y `con_datos`, igual que en TP2, para que
`relaciones-bivariadas.ipynb` (y cualquier otro notebook de TP3) pueda unir los metadatos y filtrar después.


In [ ]:
import os

os.makedirs(PROCESSED, exist_ok=True)

estaciones_central.to_file(PROCESSED + 'estaciones_central.geojson', driver='GeoJSON')
estados_central.to_file(PROCESSED + 'estados_central.geojson', driver='GeoJSON')

print(f'Guardadas {len(estaciones_central)} estaciones ({estaciones_central["con_datos"].sum()} con observaciones) '
      f'y {len(estados_central)} estados en {PROCESSED}')


## 7. Exportación del mapa y la tabla para la presentación

TP2 exportaba además una tabla de "estaciones reasignadas" (las que no tenían estado y se les asignó uno por
polígono o cercanía). Acá se omite porque no aplica: ninguna estación de TP3 necesitó esa reasignación (sección 2).


In [ ]:
OUT = '../data/out/'
os.makedirs(OUT, exist_ok=True)

# Mapa: PNG en alta resolución (se pega directo en PowerPoint)
fig.savefig(OUT + 'mapa_estaciones.png', dpi=300, bbox_inches='tight', facecolor='white')

# Tabla con encabezados listos para presentar
tabla_estados = (
    detalle_estados.reset_index()
    .rename(columns={
        'estado': 'Abrev.', 'nombre_estado': 'Estado', 'estaciones': 'Estaciones',
        'excluidas': 'Excluidas', 'porcentaje': '%', 'area_miles_km2': 'Área (miles km²)',
        'estaciones_cada_10mil_km2': 'Estaciones / 10.000 km²',
    })
)
tabla_estados = pd.concat([
    tabla_estados,
    pd.DataFrame([{'Abrev.': '', 'Estado': 'Total', 'Estaciones': tabla_estados['Estaciones'].sum(),
                   'Excluidas': tabla_estados['Excluidas'].sum(), '%': 100.0,
                   'Área (miles km²)': tabla_estados['Área (miles km²)'].sum().round(1),
                   'Estaciones / 10.000 km²': round(tabla_estados['Estaciones'].sum() / area_km2.sum() * 1e4, 2)}]),
], ignore_index=True)
tabla_estados


In [ ]:
def tabla_a_png(df, archivo, titulo):
    """Dibuja un DataFrame como tabla y la guarda como PNG."""
    alto_fila = 0.3
    fig_t, ax_t = plt.subplots(figsize=(8, alto_fila * (len(df) + 1) + 0.6))
    ax_t.set_axis_off()
    tabla = ax_t.table(
        cellText=df.astype(str).values, colLabels=df.columns,
        loc='upper center', cellLoc='center',
    )
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(10)
    tabla.scale(1, 1.4)
    tabla.auto_set_column_width(list(range(df.shape[1])))
    for (fila, col), celda in tabla.get_celld().items():
        celda.set_edgecolor('#d9d8d4')
        if fila == 0:
            celda.set_facecolor('#2a78d6')
            celda.set_text_props(color='white', fontweight='bold')
        elif fila % 2 == 0:
            celda.set_facecolor('#f4f4f2')
        if fila > 0 and df.iloc[fila - 1, 1] == 'Total':
            celda.set_text_props(fontweight='bold')
    ax_t.set_title(titulo, fontsize=12, loc='left', pad=8)
    fig_t.savefig(archivo, dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig_t)


tabla_a_png(tabla_estados, OUT + 'tabla_estaciones_por_estado.png',
            'Estaciones meteorológicas con observaciones por estado (TP3)')

# CSV editable (separador ';' y coma decimal para abrir directo en Excel en español)
tabla_estados.to_csv(OUT + 'tabla_estaciones_por_estado.csv', sep=';', decimal=',', index=False, encoding='utf-8-sig')

sorted(os.listdir(OUT))
